In [5]:
from pathlib import Path

repo_root = Path('/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline')
analysis_root = repo_root / 'miniVI_PlaceCell_analysis_V4'
data_root = analysis_root / 'data'

host = '127.0.0.1'
port = 8054
debug = False


In [6]:
manual_spike_gui_defaults = {
    # Saved-cell behavior
    # Saved cells restore their saved parameters/results; these defaults apply to unsaved cells.
    'use_saved_cell_parameters': True,

    # General preprocessing
    'baseline_window_s': 10.0,
    'segment_duration_s': 120.0,
    'include_first_burst_spike_for_spike_height': False,

    # 1st round all-spike detection
    'spike_baseline_remove_enabled': True,
    'spike_baseline_window_ms': 51.0,
    'highpass_hz': 20.0,
    'threshold_mad': 4.0,

    # Complex burst detection
    'cb_baseline_window_s': 1.0,
    'remove_spikes_for_vm': False,
    'vm_median_window_ms': 21.0,
    'vm_crossing_threshold': 0.15,
    'refine_cb_onset': True,
    'cb_onset_threshold': 0.1,
    'cb_offset_threshold': 0.2,
    'cb_max_onset_lead_ms': 50.0,
    'cb_amp_threshold': 0.6,
    'cb_duration_threshold_ms': 20.0,
    'cb_min_spikes': 0,
    'cb_isi_threshold_ms': 20.0,
    'cb_require_min_isi': False,
    'spike_height_min_isolated_spikes': 5,
    
    # 2nd round spike detection
    'second_round_ss_threshold_mad': 5.0,
    'second_round_cs_threshold_mad': 4.0,
    # Optional simple-spike height rejection
    'second_round_refine_simple_spikes_by_height': True,
    'second_round_simple_spike_min_height_fraction': 0.9,

    # 2nd round complex burst detection
    'second_round_cb_amp_threshold': 0.6,
    'second_round_cb_min_spikes': 2,

    # Plateau detection
    'plateau_baseline_window_s': 10.0,
    'plateau_vm_median_window_ms': 51.0,
    'plateau_vm_crossing_threshold': 0.25,
    'plateau_onset_threshold': 0.25,
    'plateau_offset_threshold': 0.3,
    'plateau_amp_threshold': 1.2,
    'plateau_peak_fraction_threshold': 0.8,
    'plateau_peak_fraction_duration_ms': 20.0,
    'plateau_duration_threshold_ms': 100.0,
    'plateau_min_spikes': 0,

    # Display
    'right_panel_mode': 'spike_height_normalized',
    'segment_height_px': 200,
}


In [7]:
import json
import os
import signal
import subprocess
import sys
import time
import webbrowser

if not data_root.is_dir():
    raise ValueError(f'Invalid data_root: {data_root}')

app_script = analysis_root / 'dash_manual_spike_detection_app' / 'app.py'
if not app_script.is_file():
    raise FileNotFoundError(f'Cannot find app.py at: {app_script}')
if 'manual_spike_gui_defaults' not in globals():
    raise NameError('Run the manual_spike_gui_defaults cell before launching the GUI.')

def _manual_spike_pids_to_stop():
    pids = set()
    if '_manual_spike_gui_proc' in globals() and _manual_spike_gui_proc is not None and _manual_spike_gui_proc.poll() is None:
        pids.add(int(_manual_spike_gui_proc.pid))

    port_result = subprocess.run(
        ['lsof', f'-tiTCP:{int(port)}', '-sTCP:LISTEN'],
        text=True,
        capture_output=True,
        check=False,
    )
    for token in port_result.stdout.split():
        try:
            pids.add(int(token))
        except ValueError:
            pass

    app_result = subprocess.run(
        ['pgrep', '-f', 'dash_manual_spike_detection_app/app.py'],
        text=True,
        capture_output=True,
        check=False,
    )
    for token in app_result.stdout.split():
        try:
            pid = int(token)
        except ValueError:
            continue
        if pid != os.getpid():
            pids.add(pid)
    return sorted(pid for pid in pids if pid != os.getpid())

def _pid_is_alive(pid):
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return False
    except PermissionError:
        return True
    return True

def _stop_existing_manual_spike_gui():
    pids = _manual_spike_pids_to_stop()
    if not pids:
        print('No existing Manual Spike Detection GUI processes found.')
        return
    print('Stopping existing Manual Spike Detection GUI process(es): ' + ', '.join(str(pid) for pid in pids))
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        except PermissionError as exc:
            print(f'Could not stop PID {pid}: {exc}')
    deadline = time.time() + 5.0
    remaining = [pid for pid in pids if _pid_is_alive(pid)]
    while remaining and time.time() < deadline:
        time.sleep(0.2)
        remaining = [pid for pid in remaining if _pid_is_alive(pid)]
    for pid in remaining:
        try:
            os.kill(pid, signal.SIGKILL)
            print(f'Force-stopped PID {pid}.')
        except ProcessLookupError:
            pass
        except PermissionError as exc:
            print(f'Could not force-stop PID {pid}: {exc}')
    time.sleep(0.5)

browser_host = '127.0.0.1' if host in ('0.0.0.0', '::', '') else host
url = f'http://{browser_host}:{int(port)}'

_stop_existing_manual_spike_gui()
if '_manual_spike_gui_log' in globals() and _manual_spike_gui_log is not None and not _manual_spike_gui_log.closed:
    _manual_spike_gui_log.close()

_manual_spike_defaults_path = analysis_root / f'manual_spike_detection_defaults_{int(port)}.json'
_manual_spike_defaults_path.write_text(json.dumps(manual_spike_gui_defaults, indent=2))

cmd = [
    sys.executable,
    str(app_script),
    '--data-root', str(data_root),
    '--host', str(host),
    '--port', str(int(port)),
    '--defaults-json', str(_manual_spike_defaults_path),
    '--no-browser',
]
if debug:
    cmd.append('--debug')

_manual_spike_gui_log_path = analysis_root / f'manual_spike_detection_{int(port)}.log'
_manual_spike_gui_log = open(_manual_spike_gui_log_path, 'w')
_manual_spike_gui_proc = subprocess.Popen(
    cmd,
    cwd=str(analysis_root),
    stdout=_manual_spike_gui_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

time.sleep(1.0)
if _manual_spike_gui_proc.poll() is not None:
    _manual_spike_gui_log.flush()
    log_text = _manual_spike_gui_log_path.read_text(errors='replace')
    print(log_text[-4000:])
    raise RuntimeError(f'Manual Spike Detection GUI failed to start. See {_manual_spike_gui_log_path}')

_ = webbrowser.open(url)
print(f'Manual Spike Detection GUI running at {url}')
print(f'Log file: {_manual_spike_gui_log_path}')


No existing Manual Spike Detection GUI processes found.
Manual Spike Detection GUI running at http://127.0.0.1:8054
Log file: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/manual_spike_detection_8054.log


In [8]:
if '_manual_spike_gui_proc' in globals() and _manual_spike_gui_proc is not None and _manual_spike_gui_proc.poll() is None:
    _manual_spike_gui_proc.terminate()
    print('Stopped Manual Spike Detection GUI.')
else:
    print('Manual Spike Detection GUI is not running from this kernel.')


Stopped Manual Spike Detection GUI.
